# NB2g — AI Counter-Generation: claude-opus-4-8 (Batch API)

Generates the opus share of the AI corpus through Anthropic's **Message Batches API** (async,
50% cheaper). Same prompt, gate, and normalization as my synchronous generators — only the delivery
mechanism differs.

**Input:** `fact_cards.parquet` (+ prior outputs, to skip already-processed cards).
**Output:** `aig_opus.parquet`.

## Two-run design (Batch is asynchronous)

Batch doesn't return results immediately — I submit, then collect later (usually < 1h, up to 24h).
So this notebook has two modes, switched by a single variable `BATCH_ID`:

- **Run 1 — SUBMIT** (`BATCH_ID = ''`): build 470 requests, submit the batch, save the returned
  `batch_id` to a file AND print it. Then stop.
- **Run 2 — COLLECT** (`BATCH_ID = 'msgbatch_...'`): I paste the id I got from run 1, and the
  notebook retrieves results, runs the gate, and writes the parquet.

If `BATCH_ID` is empty → it's run 1 (submit). If it's filled → it's run 2 (collect). Simple.

## Surplus instead of a guaranteed second batch

I submit 470 to bank ~450. If the gate leaves me anywhere in a healthy band I accept
the result as-is (no second batch); the small over/undershoot is absorbed by the next generator's
moving pointer. A second batch is only needed if rejects are unexpectedly high.

## Non-overlap

Sonnet and Opus run in parallel on DISJOINT slices of the cards that DeepSeek and Qwen haven't
touched: Sonnet takes `safe[0:470]`, Opus takes `safe[470:940]`. Rejects are ignored for now
(Qwen is still running); Gemini will gather all carry-over later.

## Config — set BATCH_ID to switch between submit (run 1) and collect (run 2)

In [1]:
!pip -q install anthropic >/dev/null 2>&1

import pandas as pd, numpy as np, json, re, os, time, random, glob
import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from kaggle_secrets import UserSecretsClient

API_KEY = UserSecretsClient().get_secret('ANTHROPIC_API_KEY')
client  = anthropic.Anthropic(api_key=API_KEY)

GEN_ID      = 'opus'
MODEL_NAME  = 'claude-opus-4-8'
N_SUCCESS   = 450
N_SEND      = 470
SAFE_OFFSET = 470
PRICE_IN, PRICE_OUT = 5.0, 25.0

CARDS_PATH  = '/kaggle/input/notebooks/bahaaqassem/nb2c-build-fact-cards/fact_cards.parquet'
OUT_DIR     = '/kaggle/working'
BATCH_META  = f'{OUT_DIR}/batch_meta_{GEN_ID}.json'

# Prior generators, only used to AVOID their cards (non-overlap). Rejects are ignored for now.
PRIOR_OUTPUTS = [
    '/kaggle/input/notebooks/bahaaqassem/nb2d-generate-deepseek/aig_deepseek.parquet',
    '/kaggle/input/datasets/bahaaqassem/aig-qwen/aig_qwen.parquet',          # may not exist yet — handled gracefully
]

# ================== THE ONE SWITCH ==================
# Run 1 (submit): leave this empty.  Run 2 (collect): paste the msgbatch_... id printed in run 1.
BATCH_ID = 'msgbatch_01YVfU9gzKRJuTLFdzNyWt6a'
# ===================================================

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True): print('   ', p)
    raise FileNotFoundError(preferred)

CARDS_PATH = find_parquet(CARDS_PATH, 'fact_cards', 'fact-cards')
cards = pd.read_parquet(CARDS_PATH)
print('cards:', cards.shape, '| mode:', 'COLLECT (run 2)' if BATCH_ID else 'SUBMIT (run 1)')

cards: (3500, 16) | mode: COLLECT (run 2)


## Prompt, number cleanup, formatting, gate — identical to my other generators

These are copied verbatim from the DeepSeek/Qwen notebooks so the AI class is defined the same way
regardless of which model wrote it.

In [2]:
STYLE_HINTS = [
    'ابدأ بفقرة استهلالية تلخّص الحدث', 'اذكر خلفية موجزة للسياق',
    'أورد ردود فعل الأطراف المعنية', 'اختم بما هو متوقّع أو منتظر',
    'استخدم بنية الهرم المقلوب', 'انسب المعلومات إلى مصادرها',
]
SYSTEM_PROMPT = (
    'أنت صحفي محترف يكتب تقارير إخبارية بالعربية الفصحى لصالح غرفة أخبار محترمة. '
    'تكتب بأسلوب صحفي رصين ودقيق، وتلتزم بالحقائق المعطاة دون إضافة معلومات من خارجها.'
)
MONTH_GROUPS = [
    {'يناير','كانون الثاني'}, {'فبراير','شباط'}, {'مارس','آذار'}, {'أبريل','نيسان'},
    {'مايو','أيار'}, {'يونيو','حزيران'}, {'يوليو','تموز'}, {'أغسطس','آب'},
    {'سبتمبر','أيلول'}, {'أكتوبر','تشرين الأول'}, {'نوفمبر','تشرين الثاني'}, {'ديسمبر','كانون الأول'},
]
AMBIG = {'كانون': 0, 'تشرين': 9}

def month_group(item):
    for i, g in enumerate(MONTH_GROUPS):
        if any(m in item for m in g): return i
    for k, i in AMBIG.items():
        if k in item: return i
    return None

def clean_numbers(nums, cap=6):
    seen, out = set(), []
    for n in nums:
        n = n.strip()
        if re.fullmatch(r'[٠-٩0-9]', n): continue
        g = month_group(n)
        if g is not None:
            if g in seen: continue
            seen.add(g)
        out.append(n)
    return out[:cap]

def build_prompt(card):
    ents  = json.loads(card['entities_for_prompt'])
    facts = json.loads(card['fact_points'])
    target = int(card['target_words'])
    parts = ['اكتب تقريراً إخبارياً بالعربية الفصحى عن الموضوع التالي.', '',
             f'الموضوع: {card["topic_core"]}', '',
             'الكيانات التي يجب أن يذكرها التقرير:', '، '.join(ents), '',
             'الحقائق الأساسية التي يجب تغطيتها:']
    for f in facts: parts.append(f'- {f}')
    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums: parts += ['', 'أدرج هذه الأرقام والتواريخ: ' + '، '.join(nums)]
    if card['has_agencies']:
        parts += ['', 'انسب المعلومات إلى: ' + '، '.join(json.loads(card['source_agencies']))]
    if card['has_quotes']:
        parts += ['', 'أدرج تصريحات منسوبة للأطراف المعنية، بصياغتك أنت.']
    hints = random.sample(STYLE_HINTS, k=random.choice([3, 4]))
    parts += ['', 'إرشادات التحرير:'] + [f'- {h}' for h in hints]
    parts += ['', f'الطول: لا يقل التقرير عن {target} كلمة ولا يزيد عن {int(target*1.12)} كلمة. '
                  f'اكتب تقريراً مكتملاً ضمن هذا النطاق.', '',
              'اكتب نص التقرير مباشرة: دون عنوان، ودون أي تنسيق (لا نجوم ** ولا رموز تنسيق)، '
              'ودون مقدمة أو تعليق منك.']
    return '\n'.join(parts)

def normalize_format(text):
    t = str(text)
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t)
    t = re.sub(r'__(.+?)__', r'\1', t)
    t = re.sub(r'(?<!\w)\*(.+?)\*(?!\w)', r'\1', t)
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.M)
    t = re.sub(r'^\s*[-–—>]\s+', '', t, flags=re.M)
    t = re.sub(r'^\s*[-*_]{3,}\s*$', '', t, flags=re.M)
    t = re.sub(r'\n+', ' ', t)
    t = re.sub(r'\s{2,}', ' ', t)
    return t.strip()

_AR_DIGITS = str.maketrans('٠١٢٣٤٥٦٧٨٩',
                           '0123456789')
def _norm(s):
    s = re.sub(r'[\u064B-\u0652]', '', s); s = s.translate(_AR_DIGITS)
    return (s.replace('أ','ا').replace('إ','ا').replace('آ','ا')
             .replace('ة','ه').replace('ى','ي'))
def _norm_entity(e):
    e = _norm(e)
    e = re.sub(r'^[وفبكل]?ال', '', e)
    e = re.sub(r'^لل', '', e)
    e = re.sub(r'^[وفبكل](?=.{3,})', '', e)
    return e.strip()
def _present(item, tn, is_entity=True):
    n = _norm_entity(item) if is_entity else _norm(item)
    return len(n) >= 2 and n in tn
def _number_present(item, tn):
    g = month_group(item)
    if g is not None:
        names = MONTH_GROUPS[g] | {k for k, v in AMBIG.items() if v == g}
        return any(_norm(m) in tn for m in names)
    return _present(item, tn, is_entity=False)

def acceptance_gate(article, card, W_ENT=3, W_NUM=2, W_AG=1):
    tn = _norm(article); scores, weights, detail = [], [], {}
    ents = json.loads(card['entities_for_prompt'])
    if ents:
        hit = sum(_present(e, tn) for e in ents); ent_cov = hit/len(ents)
        detail['entities'] = f'{hit}/{len(ents)}'; scores.append(ent_cov); weights.append(W_ENT)
    else:
        ent_cov = 1.0; detail['entities'] = 'none'
    if card['has_numbers']:
        nums = clean_numbers(json.loads(card['numbers_dates']))
        if nums:
            hit = sum(_number_present(n, tn) for n in nums)
            scores.append(hit/len(nums)); weights.append(W_NUM); detail['numbers'] = f'{hit}/{len(nums)}'
    if card['has_agencies']:
        ags = json.loads(card['source_agencies']); hit = sum(_present(a, tn, False) for a in ags)
        scores.append(hit/max(len(ags),1)); weights.append(W_AG); detail['agencies'] = f'{hit}/{len(ags)}'
    weighted = sum(s*w for s, w in zip(scores, weights)) / max(sum(weights), 1)
    passed = (weighted >= 0.80) and (ent_cov >= 0.65)
    return {'passed': passed, 'weighted': round(weighted, 3),
            'entity_cov': round(ent_cov, 3), 'detail': detail}

def length_ok(article, target, tol=0.15):
    n = len(article.split()); return abs(n - target)/target <= tol, n

## Work list — my disjoint slice (non-overlapping with prior + parallel generator)

I build the same ordered list, drop every card any prior generator already processed, then take my
own fixed window `safe[SAFE_OFFSET : SAFE_OFFSET + N_SEND]`. Sonnet uses offset 0, Opus uses 470, so
the two parallel batches never touch the same card.

In [3]:
ordered = cards.sample(frac=1.0, random_state=42).reset_index(drop=True)

processed = set()
for path in PRIOR_OUTPUTS:
    if not os.path.exists(path):
        kw = os.path.basename(path).replace('.parquet', '')
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True)
                if kw in p or kw.replace('_', '-') in p]
        path = hits[0] if hits else path
    if os.path.exists(path):
        processed |= set(pd.read_parquet(path)['source_pair_id'])
        print(f'prior: {os.path.basename(path)} -> {len(processed)} processed so far')
    else:
        print(f'(prior not found, skipped: {path})')

safe = ordered[~ordered['pair_id'].isin(processed)].reset_index(drop=True)
my_cards = safe.iloc[SAFE_OFFSET:SAFE_OFFSET + N_SEND].reset_index(drop=True)
print(f'{GEN_ID}: safe pool {len(safe)} | my slice safe[{SAFE_OFFSET}:{SAFE_OFFSET+N_SEND}] = {len(my_cards)} cards')
print('sample ids:', my_cards['pair_id'].head(3).tolist())

prior: aig_deepseek.parquet -> 900 processed so far
prior: aig_qwen.parquet -> 1407 processed so far
opus: safe pool 2093 | my slice safe[470:940] = 470 cards
sample ids: ['HA_03021', 'HA_03244', 'HA_02422']


## RUN 1 — submit the batch (only when BATCH_ID is empty)

Builds one request per card (a fixed decoding seed per card is baked into the prompt sampling), then
submits. The returned `msgbatch_...` id is saved to a file AND printed — copy it into `BATCH_ID` for
run 2. I map results back later by `custom_id = pair_id`.

## Smoke test — one synchronous call before submitting the batch

My first attempt errored on all 470 requests because of bad decoding params. A single live
call costs a fraction of a cent and tells me immediately whether the params are valid, so I
never burn a batch round-trip on a parameter mistake again.

In [4]:
# SMOKE TEST — validate params with ONE synchronous call before spending on a 470-request batch.
# The first batch failed 470/470 because I sent temperature+top_p together (Claude rejects that)
# and temperatures above 1.0. This cell catches any such parameter problem in seconds.
if not BATCH_ID:
    _c = my_cards.iloc[0]
    try:
        _m = client.messages.create(
            model=MODEL_NAME, max_tokens=600, system=SYSTEM_PROMPT,
            messages=[{'role': 'user', 'content': build_prompt(_c)}],
            thinking={'type': 'disabled'},   # news writing needs no reasoning; also keeps
                                             # thinking tokens from eating into max_tokens
        )
        _txt = ''.join(b.text for b in _m.content if b.type == 'text')
        print('SMOKE TEST OK — params accepted by', MODEL_NAME)
        print('  tokens:', _m.usage.input_tokens, 'in /', _m.usage.output_tokens, 'out')
        print('  sample:', _txt[:120].replace(chr(10), ' '), '...')
        SMOKE_OK = True
    except Exception as _e:
        SMOKE_OK = False
        print('SMOKE TEST FAILED — do NOT submit the batch. Real error below:')
        print(' ', type(_e).__name__, ':', str(_e)[:400])
else:
    SMOKE_OK = True

In [5]:
def sample_params():
    # Sonnet 5 / Opus 4.8 REJECT temperature, top_p and top_k (400: "temperature is deprecated
    # for this model"). Anthropic's replacement is prompt-level steering, which I already have:
    # every card samples 3-4 different STYLE_HINTS, and every card carries different content.
    # So decoding diversity here comes from the prompt, not from sampling knobs.
    return {}

if not BATCH_ID and SMOKE_OK:          # only submit if the smoke test passed
    random.seed(42)                       # reproducible per-card decoding params
    requests, meta = [], {}
    for _, card in my_cards.iterrows():
        p = sample_params()
        meta[card['pair_id']] = p
        requests.append(Request(
            custom_id=card['pair_id'],
            params=MessageCreateParamsNonStreaming(
                model=MODEL_NAME,
                # Arabic runs ~3.5 tok/word, and Claude's new tokenizer adds ~30% -> ~4.5 tok/word.
                # max_tokens is a ceiling, not a charge, so I size it generously to avoid truncation.
                max_tokens=min(int(int(card['target_words']) * 5.5) + 1000, 120000),
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': build_prompt(card)}],
                thinking={'type': 'disabled'},   # no reasoning traces; matches how I ran Qwen
            ),
        ))
    batch = client.messages.batches.create(requests=requests)
    with open(BATCH_META, 'w', encoding='utf-8') as f:
        json.dump({'batch_id': batch.id, 'model': MODEL_NAME, 'n_sent': len(requests),
                   'params': meta}, f, ensure_ascii=False)
    print('=' * 60)
    print(f'SUBMITTED {len(requests)} requests | status: {batch.processing_status}')
    print(f'BATCH_ID = {batch.id}')
    print('=' * 60)
    print('Saved to', BATCH_META)
    print('NEXT: wait (<1h usually), then set BATCH_ID above to this id and re-run to COLLECT.')
elif not SMOKE_OK:
    print('smoke test failed — submit skipped. Fix params first.')
else:
    print('BATCH_ID is set — skipping submit, going to COLLECT.')

BATCH_ID is set — skipping submit, going to COLLECT.


## RUN 2 — collect + gate (only when BATCH_ID is set)

Polls until the batch has ended, streams results (order not guaranteed — I match on `custom_id`),
applies `normalize_format` + the gate, and writes `aig_{GEN_ID}.parquet`. Gate-failed articles are
written with `gate_passed=False` (never dropped) so they can carry over to Gemini later.

In [6]:
def make_record(card, text, params, gate, nwords, lok, error=''):
    return {
        'id': f'AI_{GEN_ID}_{card["pair_id"]}', 'text': text, 'label': 'ai', 'generator': GEN_ID,
        'source_pair_id': card['pair_id'],
        'temperature': params.get('temperature', 0), 'top_p': params.get('top_p', 0),
        'target_words': int(card['target_words']), 'actual_words': nwords,
        'entities_injected': card['entities_for_prompt'],
        'coverage_weighted': gate.get('weighted', 0), 'coverage_entities': gate.get('entity_cov', 0),
        'gate_passed': bool(gate.get('passed', False)), 'length_ok': bool(lok), 'error': error,
    }

if BATCH_ID:
    # recover per-card decoding params saved at submit time (for the record); fall back if missing
    saved = {}
    for p in [BATCH_META] + glob.glob('/kaggle/input/**/batch_meta_*.json', recursive=True):
        if os.path.exists(p):
            try: saved = json.load(open(p, encoding='utf-8')).get('params', {}); break
            except Exception: pass

    print(f'polling {BATCH_ID} ...', flush=True)
    while True:
        b = client.messages.batches.retrieve(BATCH_ID)
        if b.processing_status == 'ended':
            break
        print('   still processing:', b.request_counts, flush=True); time.sleep(60)
    print('ended:', b.request_counts, flush=True)

    card_by_id = {c['pair_id']: c for _, c in my_cards.iterrows()}
    records, n_ok, n_rej, n_err = [], 0, 0, 0
    tin_tot = tout_tot = 0

    for res in client.messages.batches.results(BATCH_ID):
        pid = res.custom_id
        card = card_by_id.get(pid)
        if card is None:
            continue
        params = saved.get(pid, {})
        if res.result.type != 'succeeded':
            emsg = ''
            try:    emsg = str(res.result.error)[:200]
            except Exception: pass
            if n_err < 3:                       # print the first few real error messages
                print(f'ERROR {pid}: {res.result.type} | {emsg}', flush=True)
            records.append(make_record(card, '', params, {}, 0, False,
                                       error=f'batch:{res.result.type}:{emsg[:100]}'))
            n_err += 1; continue
        msg = res.result.message
        tin_tot += msg.usage.input_tokens; tout_tot += msg.usage.output_tokens
        text = normalize_format(''.join(b.text for b in msg.content if b.type == 'text'))
        gate = acceptance_gate(text, card); lok, nwords = length_ok(text, int(card['target_words']))
        records.append(make_record(card, text, params, gate, nwords, lok))
        if gate['passed']: n_ok += 1
        else: n_rej += 1

    out = pd.DataFrame(records)
    CKPT = f'{OUT_DIR}/aig_{GEN_ID}.parquet'
    out.to_parquet(CKPT, index=False)

    cost = tin_tot/1e6*PRICE_IN*0.5 + tout_tot/1e6*PRICE_OUT*0.5   # Batch = 50%
    print('=' * 60)
    print(f'collected {len(out)} | passed {n_ok} | rejected {n_rej} | errored {n_err}')
    print(f'target successes {N_SUCCESS} | got {n_ok}  ->', 'OK' if n_ok >= N_SUCCESS-20 else 'LOW (consider a 2nd batch)')
    print(f'cost (batch, 50%): ${cost:.2f}')
    n_md = int(out['text'].str.contains('**', regex=False).sum())
    n_nl = int(out['text'].str.contains(chr(10), regex=False).sum())
    print(f'markdown/newlines left: {n_md}/{n_nl} (must be 0)')
    print(f'saved {CKPT} | upload as aigt-aig-{GEN_ID}')
    print('=' * 60)

polling msgbatch_01YVfU9gzKRJuTLFdzNyWt6a ...
ended: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=470)
collected 470 | passed 380 | rejected 90 | errored 0
target successes 450 | got 380  -> LOW (consider a 2nd batch)
cost (batch, 50%): $18.53
markdown/newlines left: 0/0 (must be 0)
saved /kaggle/working/aig_opus.parquet | upload as aigt-aig-opus


## Notes

- **One switch, two runs.** `BATCH_ID` empty → submit; filled → collect. The id is saved to
  `batch_meta_opus.json` and printed, so I can paste it back for run 2.
- **50% Batch discount** applies to both input and output. Sonnet 5 is on its introductory
  $2/$10 rate through Aug 31 2026, so effective Batch cost is ~$1/$5 per MTok.
- **custom_id = pair_id** maps every result back to its card; results stream in arbitrary order.
- **Rejects are written, not dropped** (`gate_passed=False`) so Gemini can salvage them later.
- **Surplus, not a forced 2nd batch:** 470 submitted to bank ~450; a 2nd batch only if
  rejects are unexpectedly high (< 450-20 passes).
- Non-overlap with the parallel Opus run is guaranteed by the fixed `SAFE_OFFSET` windows.